# 3-Branch Normal-Only Anomaly Detection
## DINOv2 + ConvNeXt + WideResNet50/PatchCore

Notebook này triển khai đúng protocol:

1. **Không train classifier Normal vs Anomaly thật** vì train chỉ có normal.
2. Chia **5-fold theo từng category**.
3. Ở mỗi fold:
   - 80% normal → xây normal memory/distribution.
   - 20% normal chưa từng nằm trong memory → **OOF normal**.
   - Từ chính 20% holdout đó tạo corruption → **synthetic anomaly validation**.
4. Chạy 3 nhánh:
   - **DINOv2**: global/semantic representation.
   - **ConvNeXt**: multi-scale hierarchical representation.
   - **WideResNet50 + PatchCore-style**: local patch anomaly.
5. Chuẩn hóa score của từng nhánh bằng statistics chỉ lấy từ normal.
6. So sánh từng nhánh và **equal-score fusion**.
7. Chọn threshold theo percentile của normal score.
8. Fit lại normal memory bằng toàn bộ train-normal rồi inference public/private.

> **Quan trọng:** synthetic anomaly chỉ là validation/stress-test. Không coi nó là ground-truth anomaly thật.

**Data loading:** notebook này map ảnh hoàn toàn bằng `sample_id`; cột `relative_path` không được dùng.

In [ ]:
# Cài thư viện nếu môi trường chưa có.
# Kaggle/Colab: bỏ comment dòng dưới nếu cần.
# !pip install -q "timm>=1.0.15" scikit-learn pandas pillow tqdm joblib

In [ ]:
import os
import gc
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from torchvision import transforms

from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import KFold
from sklearn.preprocessing import normalize as sk_normalize
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)
from tqdm.auto import tqdm
import joblib

warnings.filterwarnings("ignore")

## 1. Config

Chỉ cần sửa `TRAIN_ROOT`. Nếu muốn chạy public/private thì sửa thêm `TEST_CSV` và `TEST_ROOT`.

Cấu trúc train mong đợi:

```text
dataset/train/
├── category_01/
├── category_02/
...
└── category_06/
```

In [ ]:
SEED = 42

# ============================================================
# PATH CONFIG
# ============================================================
# Ví dụ Google Drive sau khi mount:
#
# /content/drive/MyDrive/dataset/train/
#     images/
#     train1_6.csv
#     train2_5.csv
#     train3_4.csv
#
# /content/drive/MyDrive/dataset/public_test/
#     images/
#     test.csv   (hoặc public_test.csv)
#
# /content/drive/MyDrive/dataset/private_test/
#     images/
#     test.csv   (hoặc private_test.csv)
#
# Notebook KHÔNG dùng relative_path để mở ảnh.
# Chỉ dùng sample_id + category để map tới file thật trong thư mục images.

DATASET_ROOT = Path("/content/drive/MyDrive/dataset")

TRAIN_DIR = DATASET_ROOT / "train"
TRAIN_IMAGES_DIR = TRAIN_DIR / "images"
TRAIN_CSVS = [
    TRAIN_DIR / "train1_6.csv",
    TRAIN_DIR / "train2_5.csv",
    TRAIN_DIR / "train3_4.csv",
]

PUBLIC_DIR = DATASET_ROOT / "public_test"
PUBLIC_IMAGES_DIR = PUBLIC_DIR / "images"
PUBLIC_CSV = PUBLIC_DIR / "test.csv"

PRIVATE_DIR = DATASET_ROOT / "private_test"
PRIVATE_IMAGES_DIR = PRIVATE_DIR / "images"
PRIVATE_CSV = PRIVATE_DIR / "test.csv"

OUTPUT_DIR = Path("./anomaly_3branch_outputs")
CACHE_DIR = OUTPUT_DIR / "feature_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# MODEL
# ============================================================
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

DINO_MODEL = "vit_small_patch14_dinov2.lvd142m"
CONVNEXT_MODEL = "convnext_tiny.fb_in22k"
PATCH_MODEL = "wide_resnet50_2.tv2_in1k"

PCA_DIM_DINO = 128
PCA_DIM_CONV = 128

KNN_K = 5
KNN_METRIC = "cosine"

PATCH_GRID = 7
PATCH_PROJ_DIM = 128
MEMORY_PATCHES_PER_IMAGE = 8
PATCH_NEIGHBOR_SEARCH_K = 16
PATCH_TOP_FRAC = 0.10

N_FOLDS = 5

SYNTH_TYPES = [
    "cutpaste",
    "mask",
    "duplicate",
    "local_blur",
    "scratch",
]
N_SYNTH_PER_IMAGE = 1

NORMAL_PERCENTILE = 97.5

AUTO_SELECT_PERCENTILE = False
PERCENTILE_GRID = [95.0, 97.0, 97.5, 98.0, 99.0]

USE_CACHE = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = torch.cuda.is_available()

print("DEVICE:", DEVICE)
print("timm:", timm.__version__)

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

## 2. Đọc CSV và map ảnh bằng `sample_id`

Notebook **bỏ hoàn toàn `relative_path` khi mở ảnh**.

CSV chỉ cần có:

```text
sample_id, category, relative_path
```

Trong đó:
- `sample_id`: dùng làm khóa để tìm file ảnh.
- `category`: dùng để chia category.
- `relative_path`: có thể giữ nguyên trong CSV nhưng notebook **không dùng cột này**.

Ví dụ nếu CSV có:

```text
img_0029f4b2f90a9..., category_01, train/category_01/...
```

notebook sẽ tìm file thật bằng `sample_id`, chẳng hạn:

```text
train/images/category_01/img_0029f4b2f90a9....png
```

hoặc nếu ảnh nằm phẳng:

```text
train/images/img_0029f4b2f90a9....png
```

Cơ chế tương tự được dùng cho **public_test** và **private_test**.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def build_image_index(images_root: Path):
    """
    Quét toàn bộ thư mục images và tạo map:
        stem filename -> full path

    Ví dụ:
        img_abc123.png -> key 'img_abc123'
    """
    images_root = Path(images_root)
    if not images_root.exists():
        raise FileNotFoundError(f"Không thấy thư mục images: {images_root}")

    index = {}
    duplicate_ids = []

    for p in images_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            key = p.stem

            if key in index:
                duplicate_ids.append(key)
            else:
                index[key] = str(p)

    if duplicate_ids:
        print(
            f"WARNING: có {len(set(duplicate_ids))} sample_id trùng filename stem. "
            "Notebook sẽ giữ file gặp đầu tiên."
        )

    print(f"Indexed {len(index):,} images from {images_root}")
    return index


def read_csvs(csv_paths):
    frames = []

    for csv_path in csv_paths:
        csv_path = Path(csv_path)

        if not csv_path.exists():
            raise FileNotFoundError(f"Không thấy CSV: {csv_path}")

        df = pd.read_csv(csv_path)

        required = {"sample_id", "category"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{csv_path.name} thiếu cột: {missing}")

        # Chỉ giữ các cột cần cho pipeline.
        # relative_path nếu có sẽ bị bỏ qua hoàn toàn.
        df = df[["sample_id", "category"]].copy()
        df["source_csv"] = csv_path.name

        frames.append(df)

    out = pd.concat(frames, ignore_index=True)

    if out["sample_id"].duplicated().any():
        dup = out.loc[out["sample_id"].duplicated(), "sample_id"].head(10).tolist()
        raise ValueError(
            f"sample_id bị trùng giữa các CSV train. Ví dụ: {dup}"
        )

    return out


def attach_paths_by_sample_id(df, images_root: Path, image_index=None):
    """
    Map sample_id -> ảnh thật.

    Không dùng relative_path.
    """
    if image_index is None:
        image_index = build_image_index(images_root)

    out = df.copy()
    out["path"] = out["sample_id"].astype(str).map(image_index)

    missing = out["path"].isna()

    if missing.any():
        bad = out.loc[missing, ["sample_id", "category"]].head(20)
        print("\nKhông map được một số sample_id:")
        display(bad)

        raise FileNotFoundError(
            f"Không tìm thấy {missing.sum()} / {len(out)} ảnh trong {images_root}. "
            "Kiểm tra filename ảnh có đúng bằng sample_id + extension hay không."
        )

    return out


# ============================================================
# TRAIN
# ============================================================
train_meta = read_csvs(TRAIN_CSVS)
train_image_index = build_image_index(TRAIN_IMAGES_DIR)
train_df = attach_paths_by_sample_id(
    train_meta,
    TRAIN_IMAGES_DIR,
    image_index=train_image_index,
)

display(train_df.head())
display(train_df.groupby("category").size().rename("n_normal").to_frame())

print("Total train:", len(train_df))
print("CSV files:", train_df["source_csv"].value_counts().to_dict())

### Kiểm tra mapping ID → ảnh

Cell dưới xác nhận filename stem của ảnh đúng với `sample_id`. Nếu tất cả đúng, bạn có thể yên tâm notebook không dựa vào `relative_path`.

In [ ]:
check = train_df.sample(min(10, len(train_df)), random_state=SEED).copy()
check["image_stem"] = check["path"].map(lambda x: Path(x).stem)
check["id_match"] = check["sample_id"].astype(str) == check["image_stem"]

display(check[["sample_id", "category", "image_stem", "id_match", "path"]])

assert check["id_match"].all(), (
    "Có ảnh filename không khớp sample_id. "
    "Nếu dataset đặt filename khác sample_id, cần sửa hàm build_image_index."
)

print("Mapping sample_id -> image OK.")

## 3. Image preprocessing

Cả 3 backbone dùng input 224×224 + ImageNet normalization để việc cache đơn giản và công bằng hơn.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def open_rgb(path):
    with Image.open(path) as im:
        return im.convert("RGB")

## 4. Synthetic anomaly generator

Không dùng flip/rotation nhẹ làm anomaly. Ở đây corruption cố tình phá **local structure / texture**.

Mỗi synthetic image được sinh **deterministic** theo `seed`, nên chạy lại cho kết quả giống nhau.

In [ ]:
def _rand_box(rng, w, h, min_frac=0.12, max_frac=0.35):
    bw = max(4, int(rng.uniform(min_frac, max_frac) * w))
    bh = max(4, int(rng.uniform(min_frac, max_frac) * h))
    x1 = int(rng.integers(0, max(1, w - bw)))
    y1 = int(rng.integers(0, max(1, h - bh)))
    return x1, y1, x1 + bw, y1 + bh

def synthetic_corrupt(img: Image.Image, kind: str, seed: int) -> Image.Image:
    rng = np.random.default_rng(seed)
    img = img.copy().convert("RGB")
    w, h = img.size

    if kind == "cutpaste":
        sx1, sy1, sx2, sy2 = _rand_box(rng, w, h, 0.10, 0.28)
        patch = img.crop((sx1, sy1, sx2, sy2))
        if rng.random() < 0.5:
            patch = patch.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        dx = int(rng.integers(0, max(1, w - patch.width)))
        dy = int(rng.integers(0, max(1, h - patch.height)))
        img.paste(patch, (dx, dy))
        return img

    if kind == "mask":
        x1, y1, x2, y2 = _rand_box(rng, w, h, 0.08, 0.25)
        arr = np.asarray(img).astype(np.float32)
        border = np.concatenate([
            arr[max(0,y1-3):y1, x1:x2].reshape(-1,3),
            arr[y2:min(h,y2+3), x1:x2].reshape(-1,3),
        ], axis=0) if y1 > 0 or y2 < h else arr.reshape(-1,3)
        fill = tuple(np.clip(np.median(border, axis=0), 0, 255).astype(np.uint8).tolist())
        draw = ImageDraw.Draw(img)
        draw.rectangle((x1, y1, x2, y2), fill=fill)
        return img

    if kind == "duplicate":
        sx1, sy1, sx2, sy2 = _rand_box(rng, w, h, 0.08, 0.22)
        patch = img.crop((sx1, sy1, sx2, sy2))
        dx = int(rng.integers(0, max(1, w - patch.width)))
        dy = int(rng.integers(0, max(1, h - patch.height)))
        img.paste(patch, (dx, dy))
        return img

    if kind == "local_blur":
        x1, y1, x2, y2 = _rand_box(rng, w, h, 0.12, 0.35)
        patch = img.crop((x1, y1, x2, y2))
        radius = float(rng.uniform(3.0, 8.0))
        patch = patch.filter(ImageFilter.GaussianBlur(radius=radius))
        img.paste(patch, (x1, y1))
        return img

    if kind == "scratch":
        draw = ImageDraw.Draw(img)
        n_lines = int(rng.integers(2, 7))
        px = np.asarray(img)
        med = np.median(px.reshape(-1,3), axis=0)
        if med.mean() > 128:
            base = int(rng.integers(0, 70))
        else:
            base = int(rng.integers(185, 255))
        color = (base, base, base)

        for _ in range(n_lines):
            x1 = int(rng.integers(0, w))
            y1 = int(rng.integers(0, h))
            length = int(rng.integers(max(8, w//15), max(12, w//3)))
            angle = float(rng.uniform(0, 2*np.pi))
            x2 = int(np.clip(x1 + length*np.cos(angle), 0, w-1))
            y2 = int(np.clip(y1 + length*np.sin(angle), 0, h-1))
            width = int(rng.integers(1, max(2, min(w,h)//80 + 2)))
            draw.line((x1,y1,x2,y2), fill=color, width=width)
        return img

    raise ValueError(f"Unknown synthetic type: {kind}")

# Quick visual sanity check
sample_img = open_rgb(train_df.iloc[0]["path"])
display(sample_img.resize((256,256)))
for i, kind in enumerate(SYNTH_TYPES):
    display(synthetic_corrupt(sample_img, kind, SEED+i).resize((256,256)))

## 5. Dataset + model extractors

### DINOv2 branch
Lấy intermediate representations của 4 block cuối nếu `timm` hỗ trợ; mỗi block dùng:
- CLS/prefix token
- mean patch token

rồi concatenate thành image embedding.

### ConvNeXt branch
Lấy feature maps từ nhiều stage, global-average-pool từng stage rồi concatenate.

### PatchCore-style branch
Lấy 2 intermediate feature maps từ WideResNet50, resize cùng resolution, concatenate, adaptive pool về `7×7`, sau đó random projection về 128 chiều.

In [ ]:
class ImagePathDataset(Dataset):
    def __init__(self, df, transform, synthetic=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.synthetic = synthetic

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = open_rgb(row["path"])

        if self.synthetic:
            kind = row["corruption"]
            synth_seed = int(row["synth_seed"])
            img = synthetic_corrupt(img, kind, synth_seed)

        x = self.transform(img)
        return x, idx


def build_synth_df(train_df, n_per_image=1):
    rows = []
    for idx, row in train_df.reset_index(drop=True).iterrows():
        for j in range(n_per_image):
            kind = SYNTH_TYPES[(idx + j) % len(SYNTH_TYPES)]
            rows.append({
                "source_index": idx,
                "path": row["path"],
                "category": row["category"],
                                "corruption": kind,
                "synth_seed": SEED * 100000 + idx * 101 + j,
            })
    return pd.DataFrame(rows)

synth_df = build_synth_df(train_df, N_SYNTH_PER_IMAGE)
display(synth_df.head())
display(synth_df["corruption"].value_counts())

In [ ]:
def _to_nchw(feat):
    # timm features thường NCHW; một số model có thể NHWC.
    if feat.ndim != 4:
        raise ValueError(f"Expected 4D feature map, got {feat.shape}")
    if feat.shape[1] <= 64 and feat.shape[-1] > 64:
        feat = feat.permute(0, 3, 1, 2).contiguous()
    return feat


class DINOImageExtractor(nn.Module):
    def __init__(self, model_name=DINO_MODEL):
        super().__init__()
        self.model = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            img_size=IMAGE_SIZE,
        )
        self.model.eval()

    def forward(self, x):
        # Ưu tiên 4 intermediate layers.
        if hasattr(self.model, "get_intermediate_layers"):
            outs = self.model.get_intermediate_layers(
                x,
                n=4,
                return_prefix_tokens=True,
                norm=True,
            )
            features = []
            for item in outs:
                if isinstance(item, (tuple, list)) and len(item) == 2:
                    patch_tokens, prefix_tokens = item
                    patch_mean = patch_tokens.mean(dim=1)
                    if prefix_tokens is not None and prefix_tokens.numel() > 0:
                        if prefix_tokens.ndim == 3:
                            prefix = prefix_tokens[:, 0]
                        else:
                            prefix = prefix_tokens
                    else:
                        prefix = patch_mean
                    features.append(torch.cat([prefix, patch_mean], dim=1))
                else:
                    tokens = item
                    if tokens.ndim == 3:
                        features.append(torch.cat([tokens[:,0], tokens[:,1:].mean(1)], dim=1))
                    else:
                        features.append(tokens)
            z = torch.cat(features, dim=1)
        else:
            out = self.model.forward_features(x)
            if isinstance(out, dict):
                patch = out.get("x_norm_patchtokens", None)
                cls = out.get("x_norm_clstoken", None)
                if patch is not None:
                    pm = patch.mean(1)
                    if cls is None:
                        cls = pm
                    z = torch.cat([cls, pm], dim=1)
                else:
                    vals = [v for v in out.values() if torch.is_tensor(v)]
                    z = vals[-1]
            elif out.ndim == 3:
                z = torch.cat([out[:,0], out[:,1:].mean(1)], dim=1)
            else:
                z = out

        return F.normalize(z.float(), dim=1)


class ConvNeXtMultiScaleExtractor(nn.Module):
    def __init__(self, model_name=CONVNEXT_MODEL):
        super().__init__()
        self.model = timm.create_model(
            model_name,
            pretrained=True,
            features_only=True,
            out_indices=(1, 2, 3),
        )
        self.model.eval()

    def forward(self, x):
        feats = self.model(x)
        pooled = []
        for f in feats:
            f = _to_nchw(f)
            p = F.adaptive_avg_pool2d(f, 1).flatten(1)
            p = F.normalize(p.float(), dim=1)
            pooled.append(p)
        z = torch.cat(pooled, dim=1)
        return F.normalize(z, dim=1)


class PatchCoreFeatureExtractor(nn.Module):
    def __init__(self, model_name=PATCH_MODEL, proj_dim=PATCH_PROJ_DIM):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            features_only=True,
            out_indices=(2, 3),
        )
        self.backbone.eval()

        # Lazy random projection: tạo sau khi biết channel dimension.
        self.proj_dim = proj_dim
        self.register_buffer("projection", torch.empty(0), persistent=False)

    def _ensure_projection(self, in_dim, device):
        if self.projection.numel() == 0 or self.projection.shape[0] != in_dim:
            g = torch.Generator(device="cpu")
            g.manual_seed(SEED)
            mat = torch.randn(in_dim, self.proj_dim, generator=g) / math.sqrt(self.proj_dim)
            self.projection = mat.to(device)

    def forward(self, x):
        f2, f3 = self.backbone(x)
        f2 = _to_nchw(f2)
        f3 = _to_nchw(f3)

        f3 = F.interpolate(f3, size=f2.shape[-2:], mode="bilinear", align_corners=False)
        f = torch.cat([f2, f3], dim=1)

        # Cố định số patch/image để RAM và kNN ổn định.
        f = F.adaptive_avg_pool2d(f, (PATCH_GRID, PATCH_GRID))
        patches = f.flatten(2).transpose(1, 2).contiguous()  # [B, P, C]

        self._ensure_projection(patches.shape[-1], patches.device)
        patches = patches @ self.projection
        patches = F.normalize(patches.float(), dim=-1)
        return patches

## 6. Feature extraction + cache

Đây là bước tốn GPU nhất. Sau khi cache xong, bạn có thể đổi:
- số fold
- PCA dimension
- kNN K
- threshold percentile
- fusion

mà **không cần chạy lại backbone**.

In [ ]:
@torch.inference_mode()
def extract_image_features(model, df, synthetic=False, desc="extract"):
    ds = ImagePathDataset(df, eval_transform, synthetic=synthetic)
    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    model = model.to(DEVICE).eval()
    out = [None] * len(ds)

    for x, idx in tqdm(dl, desc=desc):
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            z = model(x)
        z = z.detach().float().cpu().numpy()
        for k, original_idx in enumerate(idx.numpy()):
            out[original_idx] = z[k]

    return np.stack(out)


@torch.inference_mode()
def extract_patch_features(model, df, synthetic=False, desc="patch"):
    ds = ImagePathDataset(df, eval_transform, synthetic=synthetic)
    dl = DataLoader(
        ds,
        batch_size=max(1, BATCH_SIZE // 2),
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    model = model.to(DEVICE).eval()
    out = [None] * len(ds)

    for x, idx in tqdm(dl, desc=desc):
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            p = model(x)
        p = p.detach().float().cpu().numpy()
        for k, original_idx in enumerate(idx.numpy()):
            out[original_idx] = p[k].astype(np.float32)

    return np.stack(out)  # [N, PATCH_GRID^2, PATCH_PROJ_DIM]


def cached_array(path, fn):
    path = Path(path)
    if USE_CACHE and path.exists():
        print("Load cache:", path)
        return np.load(path)
    arr = fn()
    np.save(path, arr)
    print("Saved:", path, arr.shape)
    return arr

In [ ]:
# ---------- DINO ----------
dino = DINOImageExtractor()

dino_normal = cached_array(
    CACHE_DIR / "dino_normal.npy",
    lambda: extract_image_features(dino, train_df, False, "DINO normal")
)

dino_synth = cached_array(
    CACHE_DIR / "dino_synth.npy",
    lambda: extract_image_features(dino, synth_df, True, "DINO synthetic")
)

del dino
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("DINO:", dino_normal.shape, dino_synth.shape)

In [ ]:
# ---------- ConvNeXt ----------
conv = ConvNeXtMultiScaleExtractor()

conv_normal = cached_array(
    CACHE_DIR / "conv_normal.npy",
    lambda: extract_image_features(conv, train_df, False, "ConvNeXt normal")
)

conv_synth = cached_array(
    CACHE_DIR / "conv_synth.npy",
    lambda: extract_image_features(conv, synth_df, True, "ConvNeXt synthetic")
)

del conv
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("ConvNeXt:", conv_normal.shape, conv_synth.shape)

In [ ]:
# ---------- PatchCore-style ----------
patch_model = PatchCoreFeatureExtractor()

patch_normal = cached_array(
    CACHE_DIR / "patch_normal.npy",
    lambda: extract_patch_features(patch_model, train_df, False, "Patch normal")
)

patch_synth = cached_array(
    CACHE_DIR / "patch_synth.npy",
    lambda: extract_patch_features(patch_model, synth_df, True, "Patch synthetic")
)

del patch_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Patch:", patch_normal.shape, patch_synth.shape)

## 7. Core scoring functions

### Image-level branch
- PCA fit **chỉ trên train fold**.
- L2 normalize.
- kNN cosine distance.
- Với train-normal score dùng leave-self-out (`K+1` rồi bỏ neighbor đầu tiên).

### Patch branch
Memory lấy một số patch ngẫu nhiên từ mỗi ảnh train.
- Holdout/test: nearest normal patch.
- Train calibration: bỏ mọi memory patch có cùng `owner_id`.

In [ ]:
def fit_pca_and_knn(train_feat, pca_dim, k=KNN_K, metric=KNN_METRIC):
    n_comp = int(min(pca_dim, train_feat.shape[0] - 1, train_feat.shape[1]))
    if n_comp < 2:
        raise ValueError("Không đủ sample để PCA.")

    pca = PCA(n_components=n_comp, random_state=SEED)
    z_train = pca.fit_transform(train_feat)
    z_train = sk_normalize(z_train, norm="l2")

    nn = NearestNeighbors(
        n_neighbors=min(k + 1, len(z_train)),
        metric=metric,
        algorithm="brute",
        n_jobs=-1,
    )
    nn.fit(z_train)
    return pca, nn, z_train


def transform_pca(pca, feat):
    z = pca.transform(feat)
    return sk_normalize(z, norm="l2")


def score_image_knn(nn, z_query, k=KNN_K, self_query=False):
    if self_query:
        # z_query phải là đúng training matrix đã fit nn, cùng row order.
        # Không giả định self luôn là neighbor đầu vì duplicate feature có thể tạo tie.
        n_neighbors = min(k + 8, nn.n_samples_fit_)
        dist, ind = nn.kneighbors(z_query, n_neighbors=n_neighbors)

        out = np.empty(len(z_query), dtype=np.float64)
        for i in range(len(z_query)):
            keep = ind[i] != i
            d = dist[i][keep][:k]
            if len(d) == 0:
                d = dist[i][-1:]
            out[i] = d.mean()
        return out

    n_neighbors = min(k, nn.n_samples_fit_)
    dist, ind = nn.kneighbors(z_query, n_neighbors=n_neighbors)
    return dist.mean(axis=1)


def build_patch_memory(patch_feats, image_indices, patches_per_image=MEMORY_PATCHES_PER_IMAGE):
    mem = []
    owners = []

    for image_idx in image_indices:
        p = patch_feats[image_idx]
        rng = np.random.default_rng(SEED + int(image_idx) * 7919)
        take = min(patches_per_image, len(p))
        sel = rng.choice(len(p), size=take, replace=False)
        mem.append(p[sel])
        owners.extend([int(image_idx)] * take)

    mem = np.concatenate(mem, axis=0).astype(np.float32)
    owners = np.asarray(owners, dtype=np.int64)

    nn = NearestNeighbors(
        n_neighbors=min(PATCH_NEIGHBOR_SEARCH_K, len(mem)),
        metric="euclidean",
        algorithm="brute",
        n_jobs=-1,
    )
    nn.fit(mem)
    return nn, mem, owners


def aggregate_patch_distances(d):
    # d: [n_images, n_patches]
    top_n = max(1, int(math.ceil(d.shape[1] * PATCH_TOP_FRAC)))
    part = np.partition(d, kth=d.shape[1]-top_n, axis=1)[:, -top_n:]
    return part.mean(axis=1)


def score_patch_images(nn, patch_query, query_owner_ids=None, memory_owners=None):
    n_img, n_patch, dim = patch_query.shape
    q = patch_query.reshape(-1, dim)

    if query_owner_ids is None:
        dist, _ = nn.kneighbors(q, n_neighbors=1)
        nearest = dist[:, 0].reshape(n_img, n_patch)
        return aggregate_patch_distances(nearest)

    # Leave-owner-out cho train calibration.
    search_k = min(PATCH_NEIGHBOR_SEARCH_K, nn.n_samples_fit_)
    dist, ind = nn.kneighbors(q, n_neighbors=search_k)

    owner_per_patch = np.repeat(np.asarray(query_owner_ids), n_patch)
    nearest = np.empty(len(q), dtype=np.float32)

    for i in range(len(q)):
        owner = owner_per_patch[i]
        valid = memory_owners[ind[i]] != owner
        if valid.any():
            nearest[i] = dist[i][np.argmax(valid)]  # first True
        else:
            nearest[i] = dist[i, -1]

    nearest = nearest.reshape(n_img, n_patch)
    return aggregate_patch_distances(nearest)


def robust_params(scores):
    scores = np.asarray(scores, dtype=np.float64)
    med = np.median(scores)
    mad = np.median(np.abs(scores - med))
    scale = 1.4826 * mad
    if scale < 1e-8:
        scale = np.std(scores) + 1e-8
    return float(med), float(scale)


def robust_z(scores, med, scale):
    return (np.asarray(scores, dtype=np.float64) - med) / (scale + 1e-12)

## 8. OOF + Synthetic Validation

Điểm quan trọng của cell này:

- Holdout normal **không nằm trong memory**.
- Synthetic anomaly được tạo từ **holdout normal**, không phải train-memory image.
- Threshold ở mỗi fold được tính từ **train-normal leave-one-out scores**, không nhìn synthetic label.

In [ ]:
def metrics_binary(y_true, scores, threshold):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)

    auc = roc_auc_score(y_true, scores)
    ba = balanced_accuracy_score(y_true, pred)
    f1 = f1_score(y_true, pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    tpr = tp / max(tp + fn, 1)
    tnr = tn / max(tn + fp, 1)

    return {
        "auc": auc,
        "ba": ba,
        "f1": f1,
        "sensitivity": tpr,
        "specificity": tnr,
        "threshold": float(threshold),
    }


def run_oof_for_percentile(percentile=NORMAL_PERCENTILE, verbose=True):
    all_rows = []
    metric_rows = []

    categories = sorted(train_df["category"].unique())

    for cat in categories:
        cat_global_idx = train_df.index[train_df["category"] == cat].to_numpy()
        cat_local_n = len(cat_global_idx)

        kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

        if verbose:
            print(f"\n===== {cat}: {cat_local_n} normal images =====")

        for fold, (tr_local, va_local) in enumerate(kf.split(np.arange(cat_local_n))):
            tr_idx = cat_global_idx[tr_local]
            va_idx = cat_global_idx[va_local]

            # Synthetic rows corresponding to validation source images only.
            synth_mask = synth_df["source_index"].isin(va_idx)
            syn_rows = synth_df.index[synth_mask].to_numpy()
            syn_source = synth_df.loc[syn_rows, "source_index"].to_numpy()

            # =========================================================
            # DINO
            # =========================================================
            pca_d, nn_d, zd_tr = fit_pca_and_knn(
                dino_normal[tr_idx], PCA_DIM_DINO
            )
            sd_tr = score_image_knn(nn_d, zd_tr, self_query=True)
            sd_va = score_image_knn(
                nn_d, transform_pca(pca_d, dino_normal[va_idx]), self_query=False
            )
            sd_syn = score_image_knn(
                nn_d, transform_pca(pca_d, dino_synth[syn_rows]), self_query=False
            )

            # =========================================================
            # CONVNEXT
            # =========================================================
            pca_c, nn_c, zc_tr = fit_pca_and_knn(
                conv_normal[tr_idx], PCA_DIM_CONV
            )
            sc_tr = score_image_knn(nn_c, zc_tr, self_query=True)
            sc_va = score_image_knn(
                nn_c, transform_pca(pca_c, conv_normal[va_idx]), self_query=False
            )
            sc_syn = score_image_knn(
                nn_c, transform_pca(pca_c, conv_synth[syn_rows]), self_query=False
            )

            # =========================================================
            # PATCHCORE
            # =========================================================
            nn_p, mem_p, owner_p = build_patch_memory(patch_normal, tr_idx)

            sp_tr = score_patch_images(
                nn_p,
                patch_normal[tr_idx],
                query_owner_ids=tr_idx,
                memory_owners=owner_p,
            )
            sp_va = score_patch_images(nn_p, patch_normal[va_idx])
            sp_syn = score_patch_images(nn_p, patch_synth[syn_rows])

            # =========================================================
            # Robust normalization from TRAIN-NORMAL ONLY
            # =========================================================
            d_med, d_scale = robust_params(sd_tr)
            c_med, c_scale = robust_params(sc_tr)
            p_med, p_scale = robust_params(sp_tr)

            zd_tr_score = robust_z(sd_tr, d_med, d_scale)
            zc_tr_score = robust_z(sc_tr, c_med, c_scale)
            zp_tr_score = robust_z(sp_tr, p_med, p_scale)

            zd_va = robust_z(sd_va, d_med, d_scale)
            zc_va = robust_z(sc_va, c_med, c_scale)
            zp_va = robust_z(sp_va, p_med, p_scale)

            zd_syn = robust_z(sd_syn, d_med, d_scale)
            zc_syn = robust_z(sc_syn, c_med, c_scale)
            zp_syn = robust_z(sp_syn, p_med, p_scale)

            # Equal score fusion
            sf_tr = (zd_tr_score + zc_tr_score + zp_tr_score) / 3.0
            sf_va = (zd_va + zc_va + zp_va) / 3.0
            sf_syn = (zd_syn + zc_syn + zp_syn) / 3.0

            # Threshold = percentile TRAIN NORMAL
            thr_d = np.percentile(zd_tr_score, percentile)
            thr_c = np.percentile(zc_tr_score, percentile)
            thr_p = np.percentile(zp_tr_score, percentile)
            thr_f = np.percentile(sf_tr, percentile)

            y = np.r_[np.zeros(len(va_idx)), np.ones(len(syn_rows))]

            branch_data = {
                "DINO": (np.r_[zd_va, zd_syn], thr_d),
                "ConvNeXt": (np.r_[zc_va, zc_syn], thr_c),
                "PatchCore": (np.r_[zp_va, zp_syn], thr_p),
                "FusionMean": (np.r_[sf_va, sf_syn], thr_f),
            }

            for branch, (scores, thr) in branch_data.items():
                m = metrics_binary(y, scores, thr)
                metric_rows.append({
                    "category": cat,
                    "fold": fold,
                    "branch": branch,
                    "percentile": percentile,
                    **m,
                })

            # Save per-sample scores
            for j, global_idx in enumerate(va_idx):
                all_rows.append({
                    "category": cat,
                    "fold": fold,
                    "kind": "normal",
                    "source_index": int(global_idx),
                    "corruption": "none",
                    "dino": float(zd_va[j]),
                    "conv": float(zc_va[j]),
                    "patch": float(zp_va[j]),
                    "fusion": float(sf_va[j]),
                })

            for j, syn_idx in enumerate(syn_rows):
                all_rows.append({
                    "category": cat,
                    "fold": fold,
                    "kind": "synthetic",
                    "source_index": int(syn_source[j]),
                    "corruption": synth_df.loc[syn_idx, "corruption"],
                    "dino": float(zd_syn[j]),
                    "conv": float(zc_syn[j]),
                    "patch": float(zp_syn[j]),
                    "fusion": float(sf_syn[j]),
                })

            del pca_d, nn_d, pca_c, nn_c, nn_p, mem_p, owner_p
            gc.collect()

    return pd.DataFrame(all_rows), pd.DataFrame(metric_rows)

In [ ]:
# Có thể hơi lâu ở PatchCore kNN, nhưng backbone KHÔNG chạy lại.
oof_scores, fold_metrics = run_oof_for_percentile(NORMAL_PERCENTILE, verbose=True)

oof_scores.to_csv(OUTPUT_DIR / "oof_scores.csv", index=False)
fold_metrics.to_csv(OUTPUT_DIR / "fold_metrics.csv", index=False)

display(oof_scores.head())
display(fold_metrics.head())

## 9. Xem nhánh nào tốt nhất

Ưu tiên:
1. Mean Balanced Accuracy cao.
2. Mean AUROC cao.
3. Std giữa folds thấp.
4. Fusion phải cải thiện tương đối ổn định, không chỉ một fold.

In [ ]:
summary = (
    fold_metrics
    .groupby("branch")
    .agg(
        BA_mean=("ba", "mean"),
        BA_std=("ba", "std"),
        AUC_mean=("auc", "mean"),
        AUC_std=("auc", "std"),
        F1_mean=("f1", "mean"),
        Sens_mean=("sensitivity", "mean"),
        Spec_mean=("specificity", "mean"),
    )
    .sort_values(["BA_mean", "AUC_mean"], ascending=False)
)

display(summary)

In [ ]:
# Theo từng category
per_category = (
    fold_metrics
    .groupby(["category", "branch"])
    .agg(
        BA=("ba", "mean"),
        AUC=("auc", "mean"),
        BA_std=("ba", "std"),
    )
    .reset_index()
    .sort_values(["category", "BA"], ascending=[True, False])
)

display(per_category)

## 10. Synthetic corruption breakdown

Cell này giúp phát hiện model có đang chỉ giỏi một kiểu fake anomaly hay không.

Nếu một nhánh:
- rất tốt với `mask`
- nhưng kém mạnh ở `duplicate`, `scratch`, `cutpaste`

thì representation chưa thật sự robust.

In [ ]:
def corruption_auc_table(oof_scores):
    rows = []
    normal = oof_scores[oof_scores["kind"] == "normal"]

    for cat in sorted(oof_scores["category"].unique()):
        ncat = normal[normal["category"] == cat]

        for corr in SYNTH_TYPES:
            scat = oof_scores[
                (oof_scores["category"] == cat) &
                (oof_scores["kind"] == "synthetic") &
                (oof_scores["corruption"] == corr)
            ]
            if len(scat) == 0:
                continue

            for branch_col in ["dino", "conv", "patch", "fusion"]:
                y = np.r_[np.zeros(len(ncat)), np.ones(len(scat))]
                s = np.r_[ncat[branch_col].values, scat[branch_col].values]
                auc = roc_auc_score(y, s)
                rows.append({
                    "category": cat,
                    "corruption": corr,
                    "branch": branch_col,
                    "auc": auc,
                })
    return pd.DataFrame(rows)

corr_metrics = corruption_auc_table(oof_scores)
display(
    corr_metrics
    .groupby(["branch", "corruption"])["auc"]
    .mean()
    .unstack()
    .round(4)
)

## 11. Optional: chọn một percentile global bằng synthetic OOF

Mặc định notebook **không auto-tune** để giảm nguy cơ overfit synthetic.

Nếu bật `AUTO_SELECT_PERCENTILE=True`, chỉ chọn **một percentile chung cho tất cả category**, thay vì tune riêng từng category.

In [ ]:
def evaluate_percentile_grid(grid=PERCENTILE_GRID):
    rows = []
    cached = {}

    for p in grid:
        print(f"\n### Percentile = {p}")
        _, fm = run_oof_for_percentile(p, verbose=False)
        agg = fm.groupby("branch").agg(
            BA_mean=("ba","mean"),
            BA_std=("ba","std"),
            AUC_mean=("auc","mean"),
        ).reset_index()
        agg["percentile"] = p
        rows.append(agg)

    return pd.concat(rows, ignore_index=True)

if AUTO_SELECT_PERCENTILE:
    percentile_results = evaluate_percentile_grid()
    display(
        percentile_results.sort_values(
            ["branch","BA_mean"],
            ascending=[True, False]
        )
    )

    fusion_rows = percentile_results[percentile_results["branch"] == "FusionMean"]
    selected_percentile = float(
        fusion_rows.sort_values(
            ["BA_mean","BA_std"],
            ascending=[False, True]
        ).iloc[0]["percentile"]
    )
else:
    selected_percentile = NORMAL_PERCENTILE

print("SELECTED PERCENTILE:", selected_percentile)

## 12. Fit FINAL normal models trên 100% train

Đây **không phải neural-network training**. Các backbone vẫn frozen.

Ta fit:
- DINO PCA + kNN memory.
- ConvNeXt PCA + kNN memory.
- PatchCore patch memory.
- Robust score statistics từ full-train leave-one-out normal score.
- Final category threshold từ percentile của full-train normal fusion score.

In [ ]:
def fit_final_category_models(category, percentile):
    idx = train_df.index[train_df["category"] == category].to_numpy()

    # DINO
    pca_d, nn_d, zd = fit_pca_and_knn(dino_normal[idx], PCA_DIM_DINO)
    sd_train = score_image_knn(nn_d, zd, self_query=True)
    d_med, d_scale = robust_params(sd_train)
    zd_score = robust_z(sd_train, d_med, d_scale)

    # ConvNeXt
    pca_c, nn_c, zc = fit_pca_and_knn(conv_normal[idx], PCA_DIM_CONV)
    sc_train = score_image_knn(nn_c, zc, self_query=True)
    c_med, c_scale = robust_params(sc_train)
    zc_score = robust_z(sc_train, c_med, c_scale)

    # Patch
    nn_p, mem_p, owner_p = build_patch_memory(patch_normal, idx)
    sp_train = score_patch_images(
        nn_p,
        patch_normal[idx],
        query_owner_ids=idx,
        memory_owners=owner_p,
    )
    p_med, p_scale = robust_params(sp_train)
    zp_score = robust_z(sp_train, p_med, p_scale)

    fusion_train = (zd_score + zc_score + zp_score) / 3.0
    threshold = float(np.percentile(fusion_train, percentile))

    return {
        "category": category,
        "indices": idx,
        "pca_d": pca_d,
        "nn_d": nn_d,
        "d_med": d_med,
        "d_scale": d_scale,
        "pca_c": pca_c,
        "nn_c": nn_c,
        "c_med": c_med,
        "c_scale": c_scale,
        "nn_p": nn_p,
        "patch_memory": mem_p,
        "patch_owners": owner_p,
        "p_med": p_med,
        "p_scale": p_scale,
        "threshold": threshold,
    }


final_models = {}
for cat in sorted(train_df["category"].unique()):
    print("Fitting final:", cat)
    final_models[cat] = fit_final_category_models(cat, selected_percentile)

threshold_table = pd.DataFrame([
    {
        "category": cat,
        "threshold": obj["threshold"],
        "percentile": selected_percentile,
    }
    for cat, obj in final_models.items()
])

display(threshold_table)
threshold_table.to_csv(OUTPUT_DIR / "final_thresholds.csv", index=False)

## 12.1 Freeze & save calibration artifacts

Sau khi chọn protocol/percentile cuối cùng, lưu các object này và **không fit lại từ public/private** nếu mục tiêu là kiểm tra khả năng generalize thật.

File `final_normal_models.joblib` chứa PCA, kNN normal memories, PatchCore memory và threshold theo category.

In [ ]:
ARTIFACT_PATH = OUTPUT_DIR / "final_normal_models.joblib"

joblib.dump(
    {
        "final_models": final_models,
        "selected_percentile": selected_percentile,
        "config": {
            "seed": SEED,
            "image_size": IMAGE_SIZE,
            "dino_model": DINO_MODEL,
            "convnext_model": CONVNEXT_MODEL,
            "patch_model": PATCH_MODEL,
            "pca_dim_dino": PCA_DIM_DINO,
            "pca_dim_conv": PCA_DIM_CONV,
            "knn_k": KNN_K,
            "patch_grid": PATCH_GRID,
            "patch_proj_dim": PATCH_PROJ_DIM,
            "memory_patches_per_image": MEMORY_PATCHES_PER_IMAGE,
            "patch_top_frac": PATCH_TOP_FRAC,
            "normal_percentile": selected_percentile,
        },
    },
    ARTIFACT_PATH,
    compress=3,
)

print("Frozen calibration saved to:", ARTIFACT_PATH)

## 13. Public / Private inference bằng `sample_id`

Public và private dùng cùng một logic với train:

```text
CSV sample_id + category
        ↓
map sample_id vào thư mục images
        ↓
extract feature
        ↓
predict
```

`relative_path` nếu có trong CSV sẽ **không được sử dụng**.

In [ ]:
def load_test_df(test_csv, images_root):
    """
    Đọc public/private CSV và map ảnh bằng sample_id.
    relative_path bị bỏ qua.
    """
    test_csv = Path(test_csv)
    images_root = Path(images_root)

    if not test_csv.exists():
        raise FileNotFoundError(f"Không thấy test CSV: {test_csv}")

    df = pd.read_csv(test_csv)

    required = {"sample_id", "category"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{test_csv.name} thiếu cột: {missing}")

    # Bỏ relative_path khỏi dataframe dùng cho inference.
    df = df[["sample_id", "category"]].copy()

    image_index = build_image_index(images_root)
    df = attach_paths_by_sample_id(
        df,
        images_root,
        image_index=image_index,
    )

    return df

In [ ]:
def extract_all_test_features(test_df):
    # DINO
    dino = DINOImageExtractor()
    dino_test = extract_image_features(dino, test_df, False, "DINO test")
    del dino
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Conv
    conv = ConvNeXtMultiScaleExtractor()
    conv_test = extract_image_features(conv, test_df, False, "Conv test")
    del conv
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Patch
    patch_model = PatchCoreFeatureExtractor()
    patch_test = extract_patch_features(patch_model, test_df, False, "Patch test")
    del patch_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return dino_test, conv_test, patch_test


def predict_test(test_df, dino_test, conv_test, patch_test, final_models):
    pred_rows = []

    for cat in sorted(test_df["category"].unique()):
        obj = final_models[cat]
        idx = test_df.index[test_df["category"] == cat].to_numpy()

        # DINO
        zd = transform_pca(obj["pca_d"], dino_test[idx])
        sd = score_image_knn(obj["nn_d"], zd, self_query=False)
        sd = robust_z(sd, obj["d_med"], obj["d_scale"])

        # Conv
        zc = transform_pca(obj["pca_c"], conv_test[idx])
        sc = score_image_knn(obj["nn_c"], zc, self_query=False)
        sc = robust_z(sc, obj["c_med"], obj["c_scale"])

        # Patch
        sp = score_patch_images(obj["nn_p"], patch_test[idx])
        sp = robust_z(sp, obj["p_med"], obj["p_scale"])

        fusion = (sd + sc + sp) / 3.0
        pred = (fusion >= obj["threshold"]).astype(int)

        for j, df_idx in enumerate(idx):
            pred_rows.append({
                "_df_idx": int(df_idx),
                "sample_id": test_df.loc[df_idx, "sample_id"],
                "category": cat,
                "label": int(pred[j]),
                "dino_score": float(sd[j]),
                "conv_score": float(sc[j]),
                "patch_score": float(sp[j]),
                "fusion_score": float(fusion[j]),
                "threshold": float(obj["threshold"]),
            })

    pred_df = pd.DataFrame(pred_rows).sort_values("_df_idx").reset_index(drop=True)
    return pred_df

In [ ]:
def run_inference_split(test_csv, images_root, output_name):
    """
    Chạy một split public/private.
    """
    test_csv = Path(test_csv)
    images_root = Path(images_root)

    if not test_csv.exists():
        print(f"Skip: không thấy {test_csv}")
        return None, None

    if not images_root.exists():
        print(f"Skip: không thấy {images_root}")
        return None, None

    test_df = load_test_df(test_csv, images_root)

    print(f"\n===== {output_name} =====")
    print("Images:", len(test_df))
    display(test_df.head())

    dino_test, conv_test, patch_test = extract_all_test_features(test_df)

    pred_detail = predict_test(
        test_df,
        dino_test,
        conv_test,
        patch_test,
        final_models,
    )

    detail_path = OUTPUT_DIR / f"{output_name}_prediction_details.csv"
    submit_path = OUTPUT_DIR / f"{output_name}_output.csv"

    pred_detail.to_csv(detail_path, index=False)

    submission = pred_detail[["sample_id", "category", "label"]].copy()
    submission.to_csv(submit_path, index=False)

    display(pred_detail.head())

    display(
        pred_detail
        .groupby("category")["label"]
        .agg(["count", "sum", "mean"])
        .rename(columns={
            "sum": "n_anomaly",
            "mean": "anomaly_rate",
        })
    )

    print("Saved detail:", detail_path)
    print("Saved submit:", submit_path)

    return pred_detail, submission


# ============================================================
# PUBLIC
# ============================================================
public_pred_detail, public_submission = run_inference_split(
    PUBLIC_CSV,
    PUBLIC_IMAGES_DIR,
    "task2_public",
)


# ============================================================
# PRIVATE
# ============================================================
private_pred_detail, private_submission = run_inference_split(
    PRIVATE_CSV,
    PRIVATE_IMAGES_DIR,
    "task2_private",
)

## 14. Cách đọc kết quả để quyết định model nào extract feature tốt

### Nếu muốn biết backbone nào tốt nhất độc lập
Xem `summary`:
- `BA_mean`
- `AUC_mean`
- `BA_std`

Ví dụ:

```text
PatchCore   BA=.84 ± .02
DINO        BA=.82 ± .01
ConvNeXt    BA=.78 ± .03
```

→ PatchCore tách synthetic anomaly tốt nhất; DINO ổn định nhất.

### Nếu muốn biết 3 nhánh có bổ sung nhau không
So:

```text
DINO
ConvNeXt
PatchCore
FusionMean
```

Nếu fusion tăng ổn định ở nhiều category/fold thì ensemble có giá trị.

### Không nên kết luận chỉ từ synthetic AUROC
Synthetic anomaly không phải private anomaly thật. Hãy ưu tiên cấu hình:
- ổn định qua fold,
- ổn định qua corruption type,
- threshold không quá nhạy,
- public không cần cao nhất tuyệt đối.

### Khi nào mới thử stacking learned weights?
Chỉ sau khi `FusionMean` đã ổn.

Khi đó meta-features nên là:

```text
[dino_score, conv_score, patch_score]
```

chứ không cần concatenate hàng nghìn chiều embedding.

## 15. Recommended experiment order

| Exp | DINO | ConvNeXt | PatchCore | Fusion |
|---|---:|---:|---:|---|
| E1 | ✓ |  |  | single |
| E2 |  | ✓ |  | single |
| E3 |  |  | ✓ | single |
| E4 | ✓ | ✓ |  | mean |
| E5 | ✓ |  | ✓ | mean |
| E6 |  | ✓ | ✓ | mean |
| **E7** | ✓ | ✓ | ✓ | **equal mean** |
| E8 | ✓ | ✓ | ✓ | learned/weighted fusion |

**Đừng nhảy thẳng E8.** Nếu learned fusion thắng public nhưng không thắng OOF nhiều corruption/fold, rất dễ leaderboard-overfit.